# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display the main metadata fields
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets by their @id and show basic info
print("Available record sets in the dataset (by @id):")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"@id: {rs['@id']} | name: {rs.get('name', 'N/A')}")

# For each record set, print the fields and columns (by @id)
for rs in record_sets:
    print(f"\nRecord set: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("Fields (by @id):")
    for field in fields:
        if isinstance(field, dict):
            print(f"  - {field['@id']} (name: {field.get('name', 'N/A')})")
        else:
            print(f"  - {field}")
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    print("Columns (by @id):")
    for column in columns:
        if isinstance(column, dict):
            print(f"  - {column['@id']} (name: {column.get('name', 'N/A')})")
        else:
            print(f"  - {column}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, let's extract the first available record set
record_sets_ids = [rs['@id'] for rs in record_sets]
# Show detected record sets
print(f"Detected record sets: {record_sets_ids}")

dataframes = {}
for record_set_id in record_sets_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for {record_set_id}")

# Use the first record set for exploration
main_record_set_id = record_sets_ids[0]
print(f"\nColumns for {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a numeric field for analysis.
# Let's try to auto-detect a probable numeric field (e.g., coverage, MW, peptide_count)
df = dataframes[main_record_set_id]

# Try several likely field names by iterating
possible_numeric_fields = [
    'coverage %', 'coverage', 'MW', 'Molecular Weight', 'total_peptide_count',
    'abundance', 'modifications', 'peptide_count', 'Peptide Count'
]
numeric_field = None
for field in possible_numeric_fields:
    if field in df.columns:
        numeric_field = field
        break
if numeric_field is None:
    # fallback to first float field
    num_cols = df.select_dtypes(include='number').columns
    if len(num_cols) > 0:
        numeric_field = num_cols[0]
    else:
        numeric_field = df.columns[0]  # fallback

print(f"Numeric field used for filtering: '{numeric_field}'")

# Some records might have empty or non-numeric values; try to clean
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
df_clean = df.dropna(subset=[numeric_field])

# Example: filter by high value
threshold = df_clean[numeric_field].quantile(0.95)  # top 5% as threshold
filtered_df = df_clean[df_clean[numeric_field] > threshold]

print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group analysis by a key field
possible_group_fields = ['description', 'protein_name', 'modification', 'Sample', 'sample_id']
group_field = None
for gf in possible_group_fields:
    if gf in df_clean.columns:
        group_field = gf
        break

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by '{group_field}':")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df_clean[numeric_field], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If another numeric field exists, plot the relationship
other_numeric_fields = [c for c in df_clean.select_dtypes(include='number').columns if c != numeric_field]
if other_numeric_fields:
    plt.figure(figsize=(7, 5))
    plt.scatter(df_clean[numeric_field], df_clean[other_numeric_fields[0]], alpha=0.5)
    plt.xlabel(numeric_field)
    plt.ylabel(other_numeric_fields[0])
    plt.title(f"{numeric_field} vs {other_numeric_fields[0]}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook loaded the FAIR2 dataset schema and explored its main record sets and fields.
- We inspected numerical distributions (such as protein coverage or abundance) and their normalization, grouped by relevant attributes (if available).
- The `mlcroissant` library makes it straightforward to access, extract, and analyze structured FAIR datasets by their semantic fields via their `@id`.

To extend this analysis, consider exploring additional record sets or performing deeper domain-specific queries on the data.